# 2. Every button as a Python call

`m.buttons` mirrors XPP's main window and `m.auto.buttons` the AUTO window, including the
keyboard shortcuts. Positional arguments answer whatever the button asks, in the order XPP
asks it:

| XPP asks for | pass |
|---|---|
| a value at a prompt | a number or string |
| a choice from a pop-up menu | the key letter or the entry label |
| a multi-field dialog | a dict `{field: value}` (unset fields keep their value) |
| a file name | a string |
| YES/NO | `True` / `False` |

Unanswered questions take XPP's default, as if Enter were pressed. `m.transcript` shows what
was asked during the last call.

In [1]:
import pyxpp
m = pyxpp.load("../lecar.ode")

## The button tree

In [2]:
pyxpp.show(m.buttons)

initialconds           [I]  Integrate the equations
  range                  [R]  Integrate over a range of parameters, init data, etc
  two_par_range          [2]  Integrate over range of 2 parameters, init data, etc
  last                   [L]  Pick up from last step of previous solution
  old                    [O]  Use current initial data
  go                     [G]  Use current initial data
  mouse                  [M]  Specify initial data with mouse (answer: 'ix,iy' pixels)
  shift                  [S]  Pick up from last step and shift T to latest time
  new                    [N]  Input new initial data, one prompt per variable
  shoot                  [H]  Use the initial data from last shooting from fixed pt
  file                   [F]  Read init data from file
  formula                [U]  Type in function of 't' for t^th equation
  mice                   [I]  Repeated ICs using mouse
  dae_guess              [D]  Guess new values for DAE
  backward               [B]  In

## Simple buttons

In [3]:
m.buttons.initialconds.go()          # I G
print(len(m.data))
print(m.transcript)

601
menu    Integrate -> (G)o



In [4]:
m.buttons.numerics.total(100)        # U T 100
print(m.transcript)
m.buttons.numerics.dt(0.02)          # U D 0.02
print(m.transcript)
m.numerics["total"], m.numerics["dt"]

prompt  total : [30] -> 100



prompt  Delta t : [0.05] -> 0.02



(100.0, 0.02)

A submenu choice followed by prompts: Numerics > Method > CVODE asks for two tolerances and whether the system is banded.

In [5]:
m.buttons.numerics.method.cvode(1e-8, 1e-8)
print(m.transcript)
m.numerics["meth"], m.numerics["tol"]

menu    Method -> (C)Vode
prompt  Relative tol: [0.001] -> 1e-08
prompt  Abs. Toler: [0.001] -> 1e-08
prompt  Banded system(0/1)? [0] -> (default)



('cvode', 1e-08)

Parameters button: name, value, name, value, ..., then an empty name to stop.

In [6]:
m.buttons.parameters("iapp", 0.05, "gk", 1.8, "")
print(m.transcript)
m.pars["iapp"], m.pars["gk"]

prompt  Parameter: [] -> iapp
prompt  iapp : [0] -> 0.05
prompt  Parameter: [] -> gk
prompt  gk : [2] -> 1.8
prompt  Parameter: [] -> 



(0.05, 1.8)

## Dialogs

In [7]:
m.buttons.parameters("iapp", 0.1, "gk", 2.0, "")
m.buttons.numerics.method.runge_kutta()
m.buttons.initialconds.range({"Range over": "iapp", "Steps": 4,
                              "Start": 0.0, "End": 0.1,
                              "Reset storage": "N"})   # keep all 5 runs
print(m.transcript)
len(m.data)        # 5 runs stacked in the browser

menu    Integrate -> (R)ange
field   Range Integrate.Range over = iapp (was V)
field   Range Integrate.Steps = 4 (was 20)
field   Range Integrate.Start = 0.0 (was 0)
field   Range Integrate.End = 0.1 (was 1)
field   Range Integrate.Reset storage = N (was Y)



25005

## Yes/no questions

In [8]:
m.run(total=200, iapp=0)
m.buttons.sing_pts.go(True)          # "Print eigenvalues?" -> YES, "Draw strong sets?" -> default
print(m.transcript)
print(m.log[-400:])

menu    Equilibria -> (G)o
choice  Print eigenvalues? (YES/NO) -> YES
choice  Draw Strong Sets? (YES/NO) -> <enter> (no answer on tape)

+GL*(VL-V)+GK*W*(VK-V)-ICA)  
1:dW/dt= (LAMN(V)*(NINF(V)-W))  
cur=iapp  
ica=ica  
ik=gk*w*(v-vk)  
rate=phi  
 All formulas are valid!!
 nvar=2 naux=4 nfix=1 nmark=0 NEQ=6 NODE=7 
ICA is a duplicate name
Used 18 constants and 126 symbols 
XPPAUT 8.0 Copyright (C) 2002-now  Bard Ermentrout 
TrueColor visual:  no colormap needed

 Eigenvalues:
 1.101254  +  i  0.000000 
 0.409788  +  i  0.000000 



## Files

In [9]:
m.buttons.file.write_set("lecar_run.set")        # File > Write set
m.buttons.graphic_stuff.svg("lecar.svg")        # XPP's own plot, drawn headless
import os; sorted(f for f in os.listdir(m.workdir) if not f.startswith("."))

['lecar.svg', 'lecar_run.set']

## Raw keys

`m.press(key, *answers)` presses one key in the main window: `m.press("i", "g")` is the same as
`m.buttons.initialconds.go()`.

In [10]:
m.press("i", "g")
m.transcript

'menu    Integrate -> (G)o\n'

## Mistakes are reported, not ignored

In [11]:
try:
    m.buttons.initialconds("zz")
except pyxpp.XPPError as e:
    print(e)

menu 'Integrate' has no entry 'zz'


In [12]:
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    m.buttons.erase("nobody asked for this")
print(w[0].category.__name__)

AnswersNotUsed


In [13]:
try:
    m.buttons.file.quit(True)      # XPP calls exit(); Python survives
except pyxpp.XPPError as e:
    print(e)

XPP tried to exit (code 1)
--- transcript ---
choice  Are you sure? (YES/NO) -> YES

